# Sports English-to-Bangla Dubbing Demo

This notebook uploads a video, randomly selects 30 seconds, creates a Bangla dub, and plays the output inside Colab.


In [ ]:
# BLOCK 1 — Imports and folder setup
# This block creates a fixed working directory so later blocks can find files reliably.

from pathlib import Path
import json
import math
import random
import subprocess

from IPython.display import Video, display

WORKDIR = Path("/content/sports_bangla_dubbing_demo")
INPUT_DIR = WORKDIR / "input"
OUTPUT_DIR = WORKDIR / "output"
TMP_DIR = WORKDIR / "tmp"

for directory in [INPUT_DIR, OUTPUT_DIR, TMP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Working directory:", WORKDIR)


In [ ]:
# BLOCK 2 — Upload video from local computer
# Colab browser upload can be slow for large videos. For demo use, prefer a small compressed video.

from google.colab import files

print("Upload one video file now.")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No video uploaded.")

name = next(iter(uploaded.keys()))
src = Path(name)
VIDEO_PATH = INPUT_DIR / src.name
src.rename(VIDEO_PATH)

print("Uploaded:", VIDEO_PATH)


In [ ]:
# BLOCK 3 — Helper functions
# run() executes ffmpeg/ffprobe commands.
# ffprobe_duration() returns the length of a video/audio file in seconds.

def run(cmd, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(
        cmd,
        check=check,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

def ffprobe_duration(path):
    out = run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ]).stdout.strip()
    return float(out)

print("Helpers ready.")


In [ ]:
# BLOCK 4 — Install dependencies safely in Colab
# --no-deps avoids disturbing Colab's preinstalled numpy/pandas/click stack.

!apt-get update -qq >/dev/null
!apt-get install -y -qq ffmpeg >/dev/null

!python -m pip install -q --upgrade pip
!python -m pip install -q --no-deps \
  openai-whisper \
  tiktoken \
  more-itertools \
  transformers \
  sentencepiece \
  accelerate \
  pydub \
  gTTS

import importlib.util

for pkg in ["whisper", "transformers", "sentencepiece", "pydub", "gtts"]:
    assert importlib.util.find_spec(pkg) is not None, f"{pkg} not installed"

print("Install complete. No runtime restart needed.")


In [ ]:
# BLOCK 5 — Randomly select one 30-second clip
# No user time selection. This keeps the demo simple and avoids UI errors.

video_exts = ["*.mp4", "*.mov", "*.mkv", "*.avi", "*.webm"]

videos = []
for ext in video_exts:
    videos.extend(Path("/content").rglob(ext))

videos = [
    v for v in videos
    if "selected_30s_clip" not in v.name
    and v.is_file()
]

if not videos:
    raise FileNotFoundError("No uploaded video found under /content. Run Block 2 first.")

INPUT_VIDEO = max(videos, key=lambda p: p.stat().st_mtime)
SELECTED_VIDEO = WORKDIR / "selected_30s_clip.mp4"

duration = ffprobe_duration(INPUT_VIDEO)
clip_len = min(30.0, duration)
start = random.uniform(0, max(0.0, duration - clip_len))
end = start + clip_len

run([
    "ffmpeg", "-y",
    "-ss", str(start),
    "-t", str(clip_len),
    "-i", str(INPUT_VIDEO),
    "-c:v", "libx264",
    "-c:a", "aac",
    str(SELECTED_VIDEO)
])

print("Using video:", INPUT_VIDEO)
print(f"Random clip: {start:.2f}s to {end:.2f}s")
print("Saved:", SELECTED_VIDEO)


In [ ]:
# BLOCK 6 — Extract 16 kHz mono audio from selected clip
# Whisper works well with 16 kHz mono WAV input.

AUDIO_WAV = TMP_DIR / "selected_audio_16k_mono.wav"

if not SELECTED_VIDEO.exists():
    raise FileNotFoundError("selected_30s_clip.mp4 not found. Run Block 5 first.")

run([
    "ffmpeg", "-y",
    "-i", str(SELECTED_VIDEO),
    "-vn",
    "-ac", "1",
    "-ar", "16000",
    str(AUDIO_WAV)
])

print("Audio extracted:", AUDIO_WAV)


In [ ]:
# BLOCK 7 — Whisper transcription
# Produces English transcript segments with start/end timestamps.

import whisper

if not AUDIO_WAV.exists():
    raise FileNotFoundError("Audio file not found. Run Block 6 first.")

model = whisper.load_model("base")

result = model.transcribe(
    str(AUDIO_WAV),
    language="en",
    fp16=False
)

segments = result.get("segments", [])

if not segments:
    raise RuntimeError("No speech detected by Whisper.")

for seg in segments:
    print(f"[{seg['start']:.2f}-{seg['end']:.2f}] {seg['text'].strip()}")

SEGMENTS_JSON = TMP_DIR / "segments.json"
SEGMENTS_JSON.write_text(json.dumps(segments, indent=2), encoding="utf-8")

print("Segments saved:", SEGMENTS_JSON)


In [ ]:
# BLOCK 8 — Translate English segments to Bangla using NLLB

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEGMENTS_JSON = TMP_DIR / "segments.json"
TRANSLATED_JSON = TMP_DIR / "translated_segments.json"

if not SEGMENTS_JSON.exists():
    raise FileNotFoundError("segments.json not found. Run Block 7 first.")

segments = json.loads(SEGMENTS_JSON.read_text(encoding="utf-8"))

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

bangla_segments = []

for seg in segments:
    english = seg["text"].strip()

    inputs = tokenizer(english, return_tensors="pt", truncation=True).to(device)

    output_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("ben_Beng"),
        max_length=128
    )

    bangla = tokenizer.decode(output_tokens[0], skip_special_tokens=True).strip()

    bangla_segments.append({
        "start": seg["start"],
        "end": seg["end"],
        "english": english,
        "bangla": bangla or "নীরবতা"
    })

    print("EN:", english)
    print("BN:", bangla)
    print()

TRANSLATED_JSON.write_text(
    json.dumps(bangla_segments, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Translated segments saved:", TRANSLATED_JSON)


In [ ]:
# BLOCK 9 — Generate Bangla TTS chunks with gTTS
# One WAV file is created per translated segment.

from gtts import gTTS

TRANSLATED_JSON = TMP_DIR / "translated_segments.json"
TTS_DIR = TMP_DIR / "tts_chunks"
TTS_DIR.mkdir(parents=True, exist_ok=True)

if not TRANSLATED_JSON.exists():
    raise FileNotFoundError("translated_segments.json not found. Run Block 8 first.")

bangla_segments = json.loads(TRANSLATED_JSON.read_text(encoding="utf-8"))

chunk_paths = []

for i, seg in enumerate(bangla_segments):
    text = seg.get("bangla", "").strip() or "নীরবতা"

    mp3_path = TTS_DIR / f"chunk_{i:03d}.mp3"
    wav_path = TTS_DIR / f"chunk_{i:03d}.wav"

    gTTS(text=text, lang="bn").save(str(mp3_path))

    run([
        "ffmpeg", "-y",
        "-i", str(mp3_path),
        "-ac", "1",
        "-ar", "16000",
        str(wav_path)
    ])

    chunk_paths.append(str(wav_path))

CHUNKS_JSON = TMP_DIR / "tts_chunks.json"
CHUNKS_JSON.write_text(json.dumps(chunk_paths, indent=2), encoding="utf-8")

print("Generated TTS chunks:", len(chunk_paths))
print("Saved:", CHUNKS_JSON)


In [ ]:
# BLOCK 10 — Overlay TTS chunks on original timeline
# Each Bangla chunk is placed at the timestamp of the corresponding English segment.

from pydub import AudioSegment

TRANSLATED_JSON = TMP_DIR / "translated_segments.json"
CHUNKS_JSON = TMP_DIR / "tts_chunks.json"
DUB_WAV = TMP_DIR / "bangla_dub.wav"

bangla_segments = json.loads(TRANSLATED_JSON.read_text(encoding="utf-8"))
chunk_paths = json.loads(CHUNKS_JSON.read_text(encoding="utf-8"))

clip_dur = ffprobe_duration(SELECTED_VIDEO)

base = AudioSegment.silent(
    duration=int(math.ceil(clip_dur * 1000)),
    frame_rate=16000
).set_channels(1)

for seg, chunk in zip(bangla_segments, chunk_paths):
    audio = AudioSegment.from_wav(chunk).set_frame_rate(16000).set_channels(1)

    start_ms = int(seg["start"] * 1000)
    target_ms = max(300, int((seg["end"] - seg["start"]) * 1000))

    if len(audio) > target_ms:
        audio = audio[:target_ms]
    else:
        audio += AudioSegment.silent(duration=target_ms - len(audio), frame_rate=16000)

    base = base.overlay(audio, position=start_ms)

base.export(DUB_WAV, format="wav")

print("Dub audio:", DUB_WAV)


In [ ]:
# BLOCK 11 — Mux dubbed Bangla audio with video
# This replaces the original audio with the generated Bangla dub.

FINAL_VIDEO = OUTPUT_DIR / "bangla_dubbed_30s_demo.mp4"

if not SELECTED_VIDEO.exists():
    raise FileNotFoundError("selected_30s_clip.mp4 not found. Run Block 5.")

if not DUB_WAV.exists():
    raise FileNotFoundError("bangla_dub.wav not found. Run Block 10.")

run([
    "ffmpeg", "-y",
    "-i", str(SELECTED_VIDEO),
    "-i", str(DUB_WAV),
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "copy",
    "-c:a", "aac",
    "-shortest",
    str(FINAL_VIDEO)
])

print("Final dubbed video:", FINAL_VIDEO)


In [ ]:
# BLOCK 12 — Play final dubbed video inside Colab

if not FINAL_VIDEO.exists():
    raise FileNotFoundError("Final video not found. Run Block 11 first.")

display(Video(str(FINAL_VIDEO), embed=True, width=720))
